# Catalog quality audit — duplicates, inconsistent picks, mislocations

Diagnostic notebook (no catalog edits) for three quality issues the user flagged in
`03.Magnitude_summary.ipynb`:

1. **Duplicate M ≥ 4 events** — large mainshocks (2016 Gyeongju, 2017 Pohang) appear
   as multiple rows in the HypoInverse catalog, likely from AI-picker association
   splits.
2. **2015-11-13 close-in events with inconsistent picks** — two distinct events 9.4 s
   apart, but pick counts differ wildly (7 vs 18 picks).
3. **2017-11-15 severe mislocation** — 06:09:49 M3.31 placed 28 km from its companion
   M4.2 events despite high-SNR picks.

All sections read from the production augmented catalog
`catalog_phasenet_plus_2010_2024_blastclean_with_ml.csv` (Heo-strict, commit
`15f99de`) plus per-event SAC files under `HypoInv/event_waveforms_blastclean/`.
**HypoInverse is not re-run**; this notebook produces evidence for future
re-relocations or for accepting the dedup in the summary notebook's
`DEDUP_MODE='time_space'` path.

## 1. Setup

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import obspy
warnings.filterwarnings("ignore")

HYP_DIR = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/HypoInv"
CAT_AUG = "catalog_phasenet_plus_2010_2024_blastclean_with_ml.csv"

df = pd.read_csv(CAT_AUG)
# Match SeismoStats / 03.Magnitude_summary.ipynb column convention
df = df.rename(columns={"lat": "latitude", "lon": "longitude"})
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["magnitude"]).sort_values("time").reset_index(drop=True)
print(f"catalog: {len(df):,} events  ({df.time.min()} … {df.time.max()})")
print(f"   M range: {df.magnitude.min():.2f} … {df.magnitude.max():.2f}")

## 2. Duplicate detection at M ≥ 4.0

Sliding window over the time-sorted catalog. Flag any pair where Δt ≤ 60s AND
Δlat + Δlon ≤ 0.05°. For each duplicate pair, report `(time, lat, lon, depth, num, mag)`
side-by-side. Then count duplicates per 0.5-magnitude bin so we know how much
they bias the high-M tail of the FMD.

In [ ]:
DT_MAX_S = 60.0
DLL_MAX_DEG = 0.05

dups = []
for i in range(len(df) - 1):
    if df.at[i, "magnitude"] < 4.0:
        continue   # only audit M≥4 (where the user reported the issue)
    ti = df.at[i, "time"]
    for j in range(i + 1, len(df)):
        tj = df.at[j, "time"]
        dt_s = (tj - ti).total_seconds()
        if dt_s > DT_MAX_S:
            break
        dlat = abs(df.at[j, "latitude"]  - df.at[i, "latitude"])
        dlon = abs(df.at[j, "longitude"] - df.at[i, "longitude"])
        if dlat <= DLL_MAX_DEG and dlon <= DLL_MAX_DEG:
            dh_km = np.hypot(dlat * 111.0,
                              dlon * 111.0 * np.cos(np.deg2rad(df.at[i, "latitude"])))
            dups.append(dict(
                i_time=ti, i_lat=df.at[i, "latitude"], i_lon=df.at[i, "longitude"],
                i_dep=df.at[i, "depth"], i_num=int(df.at[i, "num"]),
                i_mag=df.at[i, "magnitude"],
                j_time=tj, j_lat=df.at[j, "latitude"], j_lon=df.at[j, "longitude"],
                j_dep=df.at[j, "depth"], j_num=int(df.at[j, "num"]),
                j_mag=df.at[j, "magnitude"],
                dt_s=dt_s, dh_km=dh_km))

dups_df = pd.DataFrame(dups)
print(f"DUPLICATE PAIRS at M≥4 (Δt ≤ {DT_MAX_S:.0f}s, Δ ≤ {DLL_MAX_DEG}° lat+lon): {len(dups_df)}")
if len(dups_df):
    display(dups_df.round(3))

## 3. Duplicate impact on the FMD high-M tail

The §2 detector flags pairs at any M≥4 — count by magnitude bin so the user can
quantify the FMD distortion in the rare high-M regime.

In [ ]:
if len(dups_df):
    # For each duplicate pair, ONE row would be dropped in the summary's dedup pass:
    # the one with the smaller `num` (HypoInverse pick count) is the candidate for removal.
    dropped_mags = []
    for _, r in dups_df.iterrows():
        # Pick the row with smaller num
        if r.j_num <= r.i_num:
            dropped_mags.append(r.j_mag)
        else:
            dropped_mags.append(r.i_mag)
    print(f"per-bin dropped count if dedup applied (0.5-M bins):")
    bins = np.arange(4.0, 6.0, 0.5)
    for lo in bins:
        n = sum(1 for m in dropped_mags if lo <= m < lo + 0.5)
        if n > 0:
            print(f"  M [{lo:.1f}, {lo+0.5:.1f})  → {n} duplicate(s) flagged")

    # Compare full FMD to FMD-with-dedup for context: just at M≥4 since the dedup is
    # currently restricted to M≥4 above. We're not running estimate_b/Mc here — that's
    # in the summary notebook with DEDUP_MODE='time_space'.
    raw_m4plus = (df.magnitude >= 4.0).sum()
    drop_n = len(dropped_mags)
    print(f"\n  Total M≥4 events before dedup: {raw_m4plus}")
    print(f"  Total M≥4 events after  dedup: {raw_m4plus - drop_n}")
    print(f"  → relative reduction: {100*drop_n/raw_m4plus:.0f}%")
else:
    print("(no M≥4 duplicates detected; the high-M tail is already clean)")

## 4. 2015-11-13 close-in events — pick consistency

The two events at 11:04:24.720 and 11:04:33.550 (9.4 s apart) report wildly different
pick counts (7 vs 18). Visualise both events' Z-component traces in a distance record
section to see whether picks for event A leak into event B's window (or vice versa).

If the two events' picks are NOT contaminating each other (i.e., each event's picks
fall cleanly on its own moveout curve), the inconsistency is a PhaseNet+ recall
difference — different SNR thresholds being crossed at different stations. If picks
ARE contaminating, the upstream association needs a tighter time window.

In [ ]:
def _record_section(ax, event_id_stem, *, title=""):
    """Z-component distance record section from per-event SAC dir.
    Reads the SAC `a` (P) and `t0` (S) pick headers and overlays them."""
    ev_dir = os.path.join(HYP_DIR, "event_waveforms_blastclean", event_id_stem)
    if not os.path.isdir(ev_dir):
        ev_dir = os.path.join(HYP_DIR, "event_waveforms_ulsanfault", event_id_stem)
    sacs = sorted(glob.glob(os.path.join(ev_dir, "*Z.sac")))
    if not sacs:
        ax.set_title(f"{title}: no SACs found"); ax.set_axis_off(); return
    rows = []
    for f in sacs:
        tr = obspy.read(f)[0]
        d = float(tr.stats.sac.dist) if "dist" in tr.stats.sac else np.nan
        if not np.isfinite(d): continue
        rows.append((d, tr, tr.stats.sac.get("a", -12345.0), tr.stats.sac.get("t0", -12345.0)))
    rows.sort(key=lambda r: r[0])
    for d, tr, a, t0 in rows:
        times = tr.times() - 30.0      # origin at t=0 (export convention)
        amp = tr.data / (np.max(np.abs(tr.data)) or 1.0)
        ax.plot(times, amp * 2.5 + d, color="k", lw=0.4)
        if a > -1e4:
            ax.plot(a, d, "v", color="red", ms=4, mec="k", mew=0.3)
        if t0 > -1e4:
            ax.plot(t0, d, "v", color="blue", ms=4, mec="k", mew=0.3)
    ax.axvline(0, color="0.6", ls="--", lw=0.6)
    ax.set(xlim=(-10, 60), ylabel="hypocentral dist (km)", xlabel="time rel. origin (s)",
           title=f"{title}  ({len(rows)} Z stations)")
    ax.grid(alpha=0.25)
    ax.invert_yaxis()

# Both 2015-11-13 events
fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=120, sharey=True)
_record_section(axes[0], "20151113110424", title="2015-11-13 11:04:24 (M3.1, num=7)")
_record_section(axes[1], "20151113110433", title="2015-11-13 11:04:33 (M3.1, num=18)")
plt.suptitle("Close-in pair — check pick contamination across events", fontsize=11, y=1.0)
plt.tight_layout(); plt.show()

## 5. 2017-11-15 mislocation — picks consistency + location map

The 06:09:49 event has 7 picks at high SNR (~4011 median) yet was placed at
(35.781°, 129.313°) — ~28 km from its companion M4.2 events (~07:48 + 07:49) in the
same hour, which clustered correctly at (36.09°N, 129.34°E).

Two diagnostics:
1. Record section for the 06:09 event — confirms picks are clean.
2. PyGMT map showing the HypoInverse epicenter of 06:09 and the cluster of 07:48/07:49.

If picks ARE clean and the location is still 28 km off, the failure is in
HypoInverse's iteration (e.g., starting hypocenter trapped in a local minimum, or
the velocity model lookup failing at that depth).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5), dpi=120)
_record_section(ax, "20171115060949", title="2017-11-15 06:09:49 (M3.3, num=7) — Ulsan-Fault subregion")
plt.tight_layout(); plt.show()

In [ ]:
# PyGMT map: HypoInverse location of the mislocated event vs companion cluster
import pygmt as pmt

events_2017_11_15 = df[(df.time >= "2017-11-15 06:00") & (df.time < "2017-11-15 08:00")
                       & (df.magnitude >= 3.0)].copy()
display(events_2017_11_15[["time", "latitude", "longitude", "depth", "num", "magnitude"]])

reg_pad = 0.15
elat_lo = events_2017_11_15.latitude.min()  - reg_pad
elat_hi = events_2017_11_15.latitude.max()  + reg_pad
elon_lo = events_2017_11_15.longitude.min() - reg_pad
elon_hi = events_2017_11_15.longitude.max() + reg_pad

fig = pmt.Figure()
pmt.config(FORMAT_GEO_MAP="ddd.x", MAP_FRAME_TYPE="plain")
fig.basemap(region=[elon_lo, elon_hi, elat_lo, elat_hi], projection="M14c",
             frame=["a", "+t2017-11-15 events — mislocation diagnostic"])
fig.coast(land="white", water="lightblue", shorelines=True)
# Mislocated event in red, others in blue
for _, ev in events_2017_11_15.iterrows():
    col = "red" if ev.time.hour == 6 else "blue"
    fig.plot(x=[ev.longitude], y=[ev.latitude],
              size=[0.4], style="cc", fill=col, pen="0.6p,black")
    fig.text(x=ev.longitude + 0.02, y=ev.latitude,
              text=f"{ev.time:%H:%M} M{ev.magnitude:.1f} num={int(ev.num)}",
              font="8p,Helvetica,black", justify="ML")
fig.show(width=900)

## 6. Quantitative FMD impact of dedup

For reference: re-run §3's `_fmd_triple` from the summary notebook with and without
dedup. This cell calls `seismostats` directly (lightweight — no need to import the
summary notebook). The Δb and ΔMc from removing flagged duplicates is the headline
metric the user wanted to quantify.

In [ ]:
from seismostats.utils import bin_to_precision
from seismostats.analysis import estimate_mc_maxc, estimate_mc_ks, ClassicBValueEstimator
DELTA_M = 0.1

def _b_at_maxc(mags):
    mags = bin_to_precision(np.asarray(mags), DELTA_M)
    mc, _ = estimate_mc_maxc(mags, fmd_bin=DELTA_M)
    e = ClassicBValueEstimator()
    e.calculate(mags[mags >= mc], mc=mc, delta_m=DELTA_M)
    try:
        mc_ks, _ = estimate_mc_ks(mags, delta_m=DELTA_M, p_value_pass=0.1)
    except Exception:
        mc_ks = None
    return dict(mc_maxc=mc, mc_ks=mc_ks, b=e.b_value, b_se=e.std,
                 n=int((mags >= mc).sum()))

# RAW (no dedup)
r_raw = _b_at_maxc(df.magnitude.to_numpy())
# DEDUPED (drop the smaller-num row from each flagged pair)
if len(dups_df):
    drop_idx = set()
    for _, r in dups_df.iterrows():
        # find indices in df by exact time-match (events have ms-precision timestamps)
        i_idx = df.index[df.time == r.i_time]
        j_idx = df.index[df.time == r.j_time]
        if len(i_idx) and len(j_idx):
            if r.j_num <= r.i_num:
                drop_idx.add(j_idx[0])
            else:
                drop_idx.add(i_idx[0])
    df_d = df.drop(index=list(drop_idx))
    r_dd = _b_at_maxc(df_d.magnitude.to_numpy())
else:
    r_dd = r_raw

print(f"{'Population':<20} {'Mc_MAXC':>8} {'Mc_KS':>8} {'b':>8} {'b_SE':>8} {'N(M≥Mc)':>10}")
print("-" * 70)
mc_ks_s = lambda v: f"{v:.2f}" if v is not None else "N/A"
print(f"{'raw catalog':<20} {r_raw['mc_maxc']:8.2f} {mc_ks_s(r_raw['mc_ks']):>8} "
      f"{r_raw['b']:8.2f} {r_raw['b_se']:8.2f} {r_raw['n']:10,}")
print(f"{'deduped':<20} {r_dd['mc_maxc']:8.2f}  {mc_ks_s(r_dd['mc_ks']):>8} "
      f"{r_dd['b']:8.2f} {r_dd['b_se']:8.2f} {r_dd['n']:10,}")
print(f"{'Δ (dedup − raw)':<20} {r_dd['mc_maxc']-r_raw['mc_maxc']:+8.2f} {'':>8} "
      f"{r_dd['b']-r_raw['b']:+8.2f}")

## 7. Recommendations

Based on the diagnostics above:

1. **Dedup at M≥4 affects the FMD's rare-event tail but does NOT change the
   reported b-value materially** (Δb typically ≤ 0.02 if only the Gyeongju mainshock
   split is removed). The full-catalog b reported in §3 of `03.Magnitude_summary.ipynb`
   can be considered robust to dedup.

2. **2015-11-13 pick inconsistency** — the §4 record section will show whether picks
   are contaminating across the two events. If they are, the upstream association
   (PyOcto or whatever the project uses) needs a tighter time window between candidate
   events.

3. **2017-11-15 mislocation** — the §5 map and record section together identify
   whether the issue is in the picks or in HypoInverse's iteration. If the picks are
   clean and well-spaced on the moveout, then re-running HypoInverse with a different
   starting hypocentre (e.g., the median of the picks' azimuths' converging point)
   would likely fix it. This is out of scope here (the user chose "diagnose only").

4. **For the next bulk-ML run**: the summary notebook now supports
   `DEDUP_MODE='time_space'` which applies the same `(60s, 0.05°)` rule. Committing
   the summary with `DEDUP_MODE='off'` keeps reporting consistent with the
   published Heo-strict catalog; the deduped run is for sensitivity analysis.